> 🔑 **Lösungsversion für Lehrkräfte.** Die Schülerversion im Hauptordner wird daraus mit
> `python3 material/quelle/schuelerversion_erstellen.py` erzeugt: Code zwischen `# >>> LOESUNG` und
> `# <<< LOESUNG` wird entfernt, Zeilen mit `# | ` werden zum Platzhalter für die Lernenden.

# 13 – Transfer Learning mit MobileNetV2

**Lernsituation „KI-Sichtprüfstation“ · Doppelstunde 8**

### Lernziele
Du kannst …
- die Idee des Transfer Learning erklären und von einem Training „von Null“ abgrenzen,
- ein vortrainiertes Netz als Merkmalsextraktor nutzen und einen eigenen Klassifikationskopf trainieren,
- begründen, warum die Merkmale auf dem Raspberry Pi vorab berechnet werden,
- mit Grad-CAM überprüfen, worauf das Netz bei seiner Entscheidung „schaut“.

| Symbol | Bedeutung |
|:--:|---|
| 📖 | Grundlagen lesen / interaktiv erkunden |
| ✏️ | Aufgabe: Code **selbst schreiben** (Zellen mit `# TODO`) |
| 🔍 | Experiment / Beobachtung – Ergebnisse notieren |
| 🧪 | Teilzielkontrolle |
| 🏭 | Bezug zur industriellen Fertigung |
| 🍓 | Hinweis für den Raspberry Pi |

## 📖 Die Idee
![Transfer Learning](../material/bilder/transfer_learning.svg)

**MobileNetV2** ist ein CNN, das für Smartphones und Kleinstrechner entwickelt und mit dem Datensatz *ImageNet* (1,4 Mio. Fotos, 1000 Klassen) trainiert wurde. Seine Faltungsschichten haben gelernt, Kanten, Ecken, Rundungen und Texturen zu erkennen – das ist auch für unsere Symbole und für Bauteile nützlich.

Wir „schneiden“ die ImageNet-Ausgabeschicht ab (`include_top=False`) und setzen einen **eigenen, kleinen Kopf** für unsere Klassen darauf.

🍓 **Zwei Einstellungen für den Raspberry Pi:**
- `ALPHA` steuert die Breite des Netzes. `0.35` ist sehr schnell, `1.0` ist das Originalnetz.
- `GROESSE`: MobileNetV2 hat vortrainierte Gewichte für 96, 128, 160, 192 und 224 Pixel.

🌐 Beim **ersten** Ausführen werden die vortrainierten Gewichte (2 – 14 MB) aus dem Internet geladen und in `~/.keras/models` gespeichert. Ohne Internet: Datei vorher auf einem anderen Rechner herunterladen und dorthin kopieren.

In [ ]:
# =============================================================================
# Vorbereitung: Hilfsmodul finden (klappt im Hauptordner und in loesungen/)
# =============================================================================
import sys
from pathlib import Path

ORDNER = Path.cwd() if (Path.cwd() / "ki_hilfen.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ORDNER))

import numpy as np
import cv2
import matplotlib.pyplot as plt
import ki_hilfen as kh

import time
import keras
from keras import layers
import tensorflow as tf

kh.setze_seed(42)

DATENSATZ = ORDNER / "dataset"
GROESSE = 128          # 96, 128, 160, 192 oder 224
ALPHA = 0.35           # 0.35, 0.5, 0.75 oder 1.0
FARBE = "rgb"          # MobileNetV2 erwartet Farbbilder

X, y, klassen = kh.lade_datensatz(DATENSATZ, GROESSE, FARBE)
X_train, X_val, X_test, y_train, y_val, y_test = kh.teile_datensatz(X, y)
ANZAHL_KLASSEN = len(klassen)
print(f"Training {len(X_train)} · Validierung {len(X_val)} · Test {len(X_test)} · Klassen {klassen}")

## 1 · Das vortrainierte Basisnetz
MobileNetV2 erwartet Pixelwerte zwischen −1 und +1. Die Schicht `Rescaling(1/127.5, offset=-1)` rechnet 0 … 255 entsprechend um – und gehört wieder **ins Modell**.

In [ ]:
basis = keras.applications.MobileNetV2(
    input_shape=(GROESSE, GROESSE, 3),
    alpha=ALPHA,
    include_top=False,        # ImageNet-Ausgabeschicht weglassen
    weights="imagenet",       # vortrainierte Gewichte
    pooling="avg",            # GlobalAveragePooling → ein Merkmalsvektor pro Bild
)
basis.trainable = False       # einfrieren: Gewichte werden NICHT verändert

extraktor = keras.Sequential([
    keras.Input(shape=(GROESSE, GROESSE, 3)),
    layers.Rescaling(1 / 127.5, offset=-1),
    basis,
], name="merkmalsextraktor")

print(f"Basisnetz: {len(basis.layers)} Schichten, {basis.count_params():,} Parameter (alle eingefroren)")
print("Merkmale pro Bild:", extraktor.output_shape[-1])

## 2 · 🍓 Merkmale einmal vorab berechnen
Weil das Basisnetz eingefroren ist, liefert es für dasselbe Bild **immer dieselben** Merkmale. Statt jedes Bild in jeder Epoche erneut durch MobileNetV2 zu schicken, berechnen wir die Merkmale **einmal** und trainieren danach nur noch den kleinen Kopf. Auf dem Raspberry Pi verkürzt das das Training von Stunden auf Sekunden.

Damit trotzdem Augmentierung stattfindet, erzeugen wir von den Trainingsbildern einige zufällig veränderte Kopien (`KOPIEN`).

In [ ]:
KOPIEN = 2      # zusätzliche augmentierte Varianten je Trainingsbild (0 = keine)

augmentierung = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomZoom(0.1),
    layers.RandomBrightness(0.2),
], name="augmentierung")

def merkmale(bilder, augmentieren=False):
    ds = kh.als_tf_dataset(bilder, np.zeros(len(bilder)), batch_size=32,
                           augmentierung=augmentierung if augmentieren else None)
    return extraktor.predict(ds.map(lambda b, l: b), verbose=0)

start = time.time()
M_train = [merkmale(X_train)] + [merkmale(X_train, augmentieren=True) for _ in range(KOPIEN)]
M_train = np.concatenate(M_train)
y_train_m = np.tile(y_train, KOPIEN + 1)
M_val, M_test = merkmale(X_val), merkmale(X_test)
dauer = time.time() - start
print(f"Merkmale berechnet in {dauer:.0f} s → {dauer / (len(M_train) + len(M_val) + len(M_test)) * 1000:.0f} ms pro Bild")
print("M_train:", M_train.shape, "| M_val:", M_val.shape, "| M_test:", M_test.shape)

## 3 · Den Klassifikationskopf trainieren

### ✏️ Aufgabe 1 – Kopf bauen
Baue ein `keras.Sequential`-Modell, das einen Merkmalsvektor (Länge `M_train.shape[1]`) bekommt und die Klasse vorhersagt:
`Input` → `Dropout(0.2)` → `Dense(ANZAHL_KLASSEN, activation="softmax")`.

In [ ]:
ANZAHL_MERKMALE = M_train.shape[1]

# >>> LOESUNG
# | kopf = None   # TODO
kopf = keras.Sequential([
    keras.Input(shape=(ANZAHL_MERKMALE,)),
    layers.Dropout(0.2),
    layers.Dense(ANZAHL_KLASSEN, activation="softmax"),
], name="kopf")
# <<< LOESUNG

kh.pruefe(lambda: kopf.input_shape == (None, ANZAHL_MERKMALE), "Eingang passt zu den Merkmalen",
          "Der Kopf braucht keras.Input(shape=(ANZAHL_MERKMALE,))")
kh.pruefe(lambda: kopf.output_shape == (None, ANZAHL_KLASSEN), "Ausgang passt zur Klassenzahl",
          "Die Ausgabeschicht braucht ANZAHL_KLASSEN Neuronen mit Softmax")
kh.pruefe(lambda: kopf.count_params() == ANZAHL_MERKMALE * ANZAHL_KLASSEN + ANZAHL_KLASSEN,
          f"Nur {ANZAHL_MERKMALE * ANZAHL_KLASSEN + ANZAHL_KLASSEN:,} trainierbare Parameter",
          "Der Kopf sollte nur aus Dropout und einer Dense-Schicht bestehen")

In [ ]:
kopf.compile(optimizer=keras.optimizers.Adam(1e-3),
             loss="sparse_categorical_crossentropy", metrics=["accuracy"])
start = time.time()
verlauf = kopf.fit(M_train, y_train_m, validation_data=(M_val, y_val),
                   epochs=100, batch_size=32, verbose=0,
                   callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=15,
                                                            restore_best_weights=True)])
print(f"{len(verlauf.epoch)} Epochen in {time.time() - start:.1f} s")
kh.zeige_lernkurven(verlauf)

## 4 · Das Gesamtmodell für die Prüfstation
Für den Einsatz setzen wir Basis und Kopf zu **einem** Modell zusammen, das direkt ein Bild (0 … 255) bekommt.

In [ ]:
eingang = keras.Input(shape=(GROESSE, GROESSE, 3))
x = layers.Rescaling(1 / 127.5, offset=-1)(eingang)
x = basis(x, training=False)     # BatchNorm bleibt auch beim Fine-Tuning im Inferenzmodus
ausgang = kopf(x)
gesamt = keras.Model(eingang, ausgang, name="transfer_mobilenetv2")
gesamt.compile(loss="sparse_categorical_crossentropy", metrics=["accuracy"])

_, test_acc = gesamt.evaluate(X_test.astype("float32"), y_test, verbose=0)
print(f"Test-Genauigkeit Transfer Learning: {test_acc:.1%}")
wahrsch = gesamt.predict(X_test.astype("float32"), verbose=0)
y_pred = wahrsch.argmax(1)
kh.zeige_konfusionsmatrix(y_test, y_pred, klassen, titel="Transfer Learning – Testdaten")

## 5 · 🔍 Grad-CAM: Worauf schaut das Netz?
Eine hohe Genauigkeit allein garantiert nicht, dass das Netz das **Richtige** gelernt hat. Grad-CAM färbt die Bildbereiche ein, die die Entscheidung am stärksten beeinflusst haben.

🏭 *Klassisches Beispiel aus der Praxis: Ein Modell erkennt „fehlerhaft“ zuverlässig – aber nur, weil alle Fehlerbilder an einem anderen Tag mit anderem Licht aufgenommen wurden. Grad-CAM deckt so etwas auf.*

In [ ]:
SCHICHT = "block_6_expand_relu"          # Zwischenschicht mit feiner Auflösung
grad_modell = keras.Model(basis.inputs, [basis.get_layer(SCHICHT).output, basis.output])

def grad_cam(bild_uint8):
    x = tf.convert_to_tensor(bild_uint8[np.newaxis].astype("float32") / 127.5 - 1)
    with tf.GradientTape() as tape:
        tape.watch(x)                            # sonst wird im eingefrorenen Netz nichts aufgezeichnet
        karten, merkm = grad_modell(x, training=False)
        vorhersage = kopf(merkm, training=False)
        klasse = int(tf.argmax(vorhersage[0]))
        wert = vorhersage[0, klasse]
    grad = tape.gradient(wert, karten)
    gewichte = tf.reduce_mean(grad, axis=(0, 1, 2))
    heat = tf.nn.relu(tf.reduce_sum(karten[0] * gewichte, axis=-1)).numpy()
    heat = heat / (heat.max() + 1e-8)
    heat = cv2.resize(heat, (bild_uint8.shape[1], bild_uint8.shape[0]))
    return heat, klasse, float(vorhersage[0, klasse])

# je Klasse ein Testbild
plt.figure(figsize=(3.2 * ANZAHL_KLASSEN, 6.4))
for k in range(ANZAHL_KLASSEN):
    i = np.where(y_test == k)[0][0]
    heat, klasse, p = grad_cam(X_test[i])
    farbig = cv2.applyColorMap(np.uint8(255 * heat), cv2.COLORMAP_JET)[..., ::-1]
    overlay = (0.55 * X_test[i] + 0.45 * farbig).astype("uint8")
    plt.subplot(2, ANZAHL_KLASSEN, k + 1); plt.imshow(X_test[i]); plt.axis("off"); plt.title(f"echt: {klassen[k]}")
    plt.subplot(2, ANZAHL_KLASSEN, ANZAHL_KLASSEN + k + 1); plt.imshow(overlay); plt.axis("off")
    plt.title(f"→ {klassen[klasse]} ({p:.0%})", color="green" if klasse == k else "red")
plt.tight_layout(); plt.show()

🔍 Schaut das Netz auf das Symbol oder auf etwas anderes (Papierrand, Tisch, Schatten)? Was würdet ihr an der Aufnahme ändern, wenn das Netz auf den Hintergrund achtet?

## 6 · ⭐ Zusatz: Feinabstimmung (Fine-Tuning)
Beim Fine-Tuning werden die oberen Schichten des Basisnetzes freigegeben und mit **sehr kleiner Lernrate** weitertrainiert. Das kann noch einige Prozentpunkte bringen.

🍓 Das Training läuft jetzt wieder durch das ganze Netz – auf dem Raspberry Pi 5 dauert eine Epoche mehrere Minuten. Empfehlung: nur mit Zeitpuffer oder auf einem PC ausführen. Dazu `FEINABSTIMMUNG = True` setzen.

In [ ]:
FEINABSTIMMUNG = False
AB_SCHICHT = 100                 # Schichten davor bleiben eingefroren

if FEINABSTIMMUNG:
    basis.trainable = True
    for schicht in basis.layers[:AB_SCHICHT]:
        schicht.trainable = False
    gesamt.compile(optimizer=keras.optimizers.Adam(1e-5),     # sehr kleine Lernrate!
                   loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    train_ds = kh.als_tf_dataset(X_train, y_train, 32, mischen=True, augmentierung=augmentierung)
    val_ds = kh.als_tf_dataset(X_val, y_val, 32)
    verlauf_fein = gesamt.fit(train_ds, validation_data=val_ds, epochs=10, verbose=2,
                              callbacks=[keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True)])
    kh.zeige_lernkurven(verlauf, verlauf_fein, markierung=len(verlauf.epoch) - 1)
    _, test_acc = gesamt.evaluate(X_test.astype("float32"), y_test, verbose=0)
    print(f"Test-Genauigkeit nach Feinabstimmung: {test_acc:.1%}")

## 7 · Speichern

In [ ]:
kh.speichere_modell(gesamt, "transfer", klassen, GROESSE, FARBE,
                    test_genauigkeit=round(float(test_acc), 4), datensatz=DATENSATZ.name, alpha=ALPHA)

### 🔍 Vergleich der drei Ansätze
Tragt die Ergebnisse eurer Modelle ein. Die Inferenzzeit auf dem Raspberry Pi messt ihr in Notebook 20.

| Modell | Bild | Parameter | Test-Genauigkeit | Trainingszeit | Dateigröße (.tflite) |
|---|---|---|---|---|---|
| FCNN (11) | 32 × 32 grau | | | | |
| CNN (12) | 96 × 96 RGB | | | | |
| Transfer (13) | 128 × 128 RGB | | | | |

Welches Modell würdet ihr für die Prüfstation wählen – und warum?

In [ ]:
kh.ram_aufraeumen()